# Tiering Model Training (Kaggle)

This notebook trains the Tier-1/Tier-2 classifier using uploaded artifacts (`train.pkl`, `val.pkl`).

Inputs expected on Kaggle:
- A dataset you upload containing `artifacts/tiering/train.pkl` and `artifacts/tiering/val.pkl`.
- This repo cloned for the tiering utilities.

Outputs:
- `artifacts/tiering/model.json`
- `artifacts/tiering/metrics.json`
- `artifacts/tiering/threshold.json`

Set `INPUT_DATASET` below to your dataset mount path (e.g., `/kaggle/input/tiering-artifacts`).

In [1]:
# Install minimal dependencies
!pip install -q xgboost==2.0.3 --no-deps
!pip install -q --no-deps "search_system @ git+https://github.com/timothycao/search-system.git"

# Clone the repository for tiering utilities
import os, sys, subprocess
REPO_URL = "https://github.com/timothycao/search-systems.git"
REPO_DIR = "/kaggle/working/search-systems"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

sys.path.append(REPO_DIR)

  Preparing metadata (setup.py) ... done


Cloning into '/kaggle/working/search-systems'...


In [2]:
import numpy as np
import json
from pathlib import Path
import xgboost as xgb
from systems.tiering import load_dataset, select_threshold, evaluate_at_threshold

# Configure paths
INPUT_DATASET = Path("/kaggle/input/tiering-artifacts")  # adjust to your dataset name
TRAIN_PATH = INPUT_DATASET / "train_train.pkl"
VAL_PATH = INPUT_DATASET / "val_train.pkl"

OUTPUT_DIR = Path("/kaggle/working/artifacts/tiering")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / "model.json"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
THRESHOLD_PATH = OUTPUT_DIR / "threshold.json"

# Hyperparameters
TARGET_RATIO = 0.4
NUM_ROUNDS = 500
EARLY_STOPPING = 50
LEARNING_RATE = 0.05
MAX_DEPTH = 6
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.8
SEED = 42
USE_GPU = True  # set False if no GPU

ImportError: cannot import name 'select_threshold' from 'systems.tiering' (/kaggle/working/search-systems/systems/tiering/__init__.py)

In [11]:
# Load datasets
train_ds = load_dataset(TRAIN_PATH)
val_ds = load_dataset(VAL_PATH)
feature_names = train_ds.get("feature_names", None)

dtrain = xgb.DMatrix(
    np.array(train_ds["X"], dtype=np.float32),
    label=np.array(train_ds["y"], dtype=np.float32),
    feature_names=feature_names if feature_names else None,
)
dval = xgb.DMatrix(
    np.array(val_ds["X"], dtype=np.float32),
    label=np.array(val_ds["y"], dtype=np.float32),
    feature_names=feature_names if feature_names else None,
)

# Training params: use hist + device=cuda to avoid gpu_hist deprecation warning
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": LEARNING_RATE,
    "max_depth": MAX_DEPTH,
    "subsample": SUBSAMPLE,
    "colsample_bytree": COLSAMPLE_BYTREE,
    "seed": SEED,
    "tree_method": "hist",
}
if USE_GPU:
    params["device"] = "cuda"

evals_result = {}
booster = xgb.train(
    params,
    dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=[(dtrain, "train"), (dval, "val")],
    evals_result=evals_result,
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=False,
)

metrics = {
    "best_iteration": int(booster.best_iteration),
    "best_score": float(booster.best_score),
    "eval_metric": params["eval_metric"],
    "evals_result": {k: [float(vv) for vv in v[params["eval_metric"]]] for k, v in evals_result.items()},
}

In [12]:
# Validation probabilities and threshold selection
val_probs = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
threshold = select_threshold(list(val_probs), target_ratio=TARGET_RATIO)
pr_metrics = evaluate_at_threshold(list(val_probs), val_ds["y"], threshold)

# Save outputs
booster.save_model(MODEL_PATH)

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, list):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

metrics_out = to_py({
    "training": metrics,
    "threshold": float(threshold),
    "target_ratio": float(TARGET_RATIO),
    "val_precision": float(pr_metrics["precision"]),
    "val_recall": float(pr_metrics["recall"]),
    "val_pred_ratio": float(pr_metrics["pred_ratio"]),
})

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, indent=2)

with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
    json.dump({"threshold": float(threshold), "target_ratio": float(TARGET_RATIO)}, f, indent=2)

metrics_out

{'training': {'best_iteration': 64,
  'best_score': 0.9999924764629781,
  'eval_metric': 'auc',
  'evals_result': {'train': [0.9999926054628362,
    0.9999926850541412,
    0.9999927463623189,
    0.9999928428130898,
    0.9999928928439408,
    0.9999929608846458,
    0.9999929464816427,
    0.9999929854763789,
    0.9999930090565465,
    0.9999930179494868,
    0.9999930552894678,
    0.9999930666679759,
    0.9999931092929222,
    0.9999931254224652,
    0.9999931370904858,
    0.9999931646255604,
    0.9999931623278653,
    0.999993173918813,
    0.9999929577416894,
    0.9999929762053196,
    0.999993011009933,
    0.9999930389152987,
    0.9999930670534651,
    0.9999930823693088,
    0.99999310904001,
    0.9999931159422141,
    0.9999931211717168,
    0.9999931294347499,
    0.9999931522655081,
    0.9999931953667154,
    0.9999931980315786,
    0.9999932063090187,
    0.9999932221276888,
    0.9999932263135228,
    0.999993234236078,
    0.9999932380958411,
    0.99999327504067

## After training
- Check `artifacts/tiering/` under the **Outputs** tab for `model.json`, `metrics.json`, `threshold.json`.
- Download them to use in your ingestion/query pipeline.